In [0]:
spark.version

## Creating CUSTOMERS_RAW Table

In [0]:
from pyspark.sql import functions as F

NUM_COSTUMERS = 50_000

In [0]:
# initialize rows data
customers_data = spark.range(1,NUM_COSTUMERS+1)

In [0]:
# adding columns and value data
customers_data = customers_data.withColumns({
    'customer_id' : F.format_string("C%06d", F.col('id')),
    'first_name' : F.element_at(F.array(F.lit('Ali'), F.lit('Marry'), F.lit('Muhammad'), F.lit('John'), F.lit('Abishek'), F.lit('Khairul'), F.lit('Zyan'), F.lit('Lucy'), F.lit('Chika'), F.lit('Gerrard')), (F.rand() * 10 + 1).cast('int')),
    'last_name' : F.element_at(F.array(F.lit('Zamir'), F.lit('Catherina'), F.lit('Abdullah'), F.lit('Smith'), F.lit('Rajesh'), F.lit('Khamis'), F.lit('Ryan'), F.lit('Lisa'), F.lit('Chota'), F.lit('Michael')), (F.rand() * 10 + 1).cast('int')),
    'email': F.lower(F.concat(F.col('first_name'), F.lit('.'), F.col('last_name'), F.lit('@nocodeah.com'))),
    'country': F.element_at(F.array(F.lit('Malaysia'), F.lit('UK'), F.lit('Germany'), F.lit('Australia'), F.lit('India'), F.lit('China'), F.lit('Japan')), (F.rand() * 7 + 1).cast('int')),
    'customer_segment': F.element_at(F.array(F.lit('Standard'), F.lit('Premium'), F.lit('Enterprise')), (F.rand() * 3 + 1).cast('int')),
    'signup_date': F.date_sub(F.current_date(),(F.rand() * 1000).cast("int"))
}).drop('id')

In [0]:
customers_data.count()
customers_data.printSchema()

In [0]:
customers_data.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable('customers_raw')

In [0]:
display(spark.sql('SHOW TABLES').show())
display(spark.table("customers_raw").show(10))

## Creating PRODUCTS_RAW Table

In [0]:
# intialize rows data
NUM_PRODUCTS = 5000
products_data = spark.range(1, NUM_PRODUCTS+1)

# adding columns and value data
products_data = products_data.withColumns({
    'product_id': F.format_string('P%06d', F.col('id')),
    'category' : F.element_at(F.array(F.lit("Electronics"),F.lit("Home"),F.lit("Fashion"),F.lit("Sports"),F.lit("Beauty")), (F.rand()*5+1).cast('int')),
    'subacategory':
        F.when(F.col('category')=='Electronics', 'Accessories')
        .when(F.col("category") == "Home", "Kitchen")
        .when(F.col("category") == "Fashion", "Clothing")
        .when(F.col("category") == "Sports", "Fitness")
        .otherwise("Personal Care"),
    'product_name' : F.concat(F.col("category"),F.lit(" Product "),F.col("product_id")),
    'unit_cost' : F.round(F.rand() * 200 + 5, 2),
    'unit_price' : F.round(F.col("unit_cost") * (F.rand() * 2 + 1.2),2)
}).drop('id')


In [0]:
display(products_data.groupBy('category').count().show())
display(products_data.head(10))

# check invalid product data, if unit_cost > unit_price
display(products_data.filter(F.col('unit_cost') > F.col('unit_price')).count())

In [0]:
products_data.write.format('delta').mode('overwrite').saveAsTable('products_raw')

## Data Exploratory

In [0]:
spark.sql('select * from products_raw fetch limit 10').show()

In [0]:
spark.sql("""
    SELECT
        category,
        COUNT(*) AS product_count,
        ROUND(AVG(unit_price), 2) AS avg_price
    FROM products_raw
    GROUP BY category
    ORDER BY product_count DESC
""").show()